# Missed Trash Pickups: Red River Waste Solutions
**Juan Zavala** · AI for Analytics · hubNashville service requests, Nov 2017 to Nov 2019

Claude worked as a **pair programmer** here, not an answer generator. Every step follows the same pattern: Claude explains its approach in plain English, I review that plan, and only then is code written and run.

| Label | Meaning |
|---|---|
| **[ME]** | Code I wrote myself |
| **[CLAUDE]** | Code Claude wrote after I approved its plan |
| **[REVIEW]** | What I changed or questioned in Claude's plan or code |
| **[CHECK]** | A validation step: does the code do what we think it does? |

Part 1 (planning, no code) is in [`part1_planning.md`](part1_planning.md).

In [1]:
# [ME] Load the data
import pandas as pd

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 200)

trash = pd.read_csv(
    'data/trash_hauler_report_with_lat_lng.csv',
    dtype={'Request Number': str, 'Zip Code': str, 'Trash Route': str, 'Council District': str},
)
trash['Date Opened'] = pd.to_datetime(trash['Date Opened'], format='%m/%d/%Y')

print(trash.shape)
trash.head(3)

(20226, 13)


,Request Number,Date Opened,Request,Description,Incident Address,Zip Code,Trash Hauler,Trash Route,Council District,State Plan X,State Plan Y,LONGITUDE,LATITUDE
0,25270,2017-11-01,Trash - Backdoor,"house with the wheel chair ramp, they share driveway, in back driveway. 3817 Crouch Dr near NES light post...",3817 Crouch Dr,37207,RED RIVER,3205,2,1727970.412,686779.4781,-86.815392,36.217292
1,25274,2017-11-01,Trash - Curbside/Alley Missed Pickup,Curb/Trash miss Tuesday.,4028 Clarksville Pike,37218,RED RIVER,4202,1,1721259.366,685444.7996,-86.838103,36.213470
2,25276,2017-11-01,Trash - Curbside/Alley Missed Pickup,Curb/trash miss Tuesday.,6528 Thunderbird Dr,37209,RED RIVER,4205,20,1707026.753,659887.4716,-86.885562,36.142923


In [2]:
# [ME] First look at the columns that matter
print(trash['Request'].value_counts(), '\n')
print(trash['Trash Hauler'].value_counts(dropna=False), '\n')
print('Dates:', trash['Date Opened'].min().date(), 'to', trash['Date Opened'].max().date())
print('Distinct raw addresses:', trash['Incident Address'].nunique())

Request
Trash - Curbside/Alley Missed Pickup    15028
Trash - Backdoor                         2629
Trash Collection Complaint               2312
Damage to Property                        257
Name: count, dtype: int64 

Trash Hauler
RED RIVER    14395
METRO         3512
WASTE IND     1350
NaN            901
Metro           68
Name: count, dtype: int64 

Dates: 2017-11-01 to 2019-11-01
Distinct raw addresses: 14121


---
# Part 2.1: Clean and standardize the address column

### Prompt I gave Claude
> "I'm working in Python with pandas. Help me clean and standardize the `Incident Address` column so the same house is always written the same way. **Explain your approach in plain English first. Don't write code until I approve it.** Use built-in pandas methods instead of a custom function where you can."

### Claude's approach (before any code)
1. **Upper-case everything**, so "Dr" and "DR" match.
2. **Keep only the street part.** Many addresses end in `, NASHVILLE, TN 37205, UNITED STATES` and many don't, so cut everything after the first comma.
3. **Remove punctuation noise.** Drop periods (`RD.` becomes `RD`) and turn `#` into a space (`#416` becomes `416`).
4. **Collapse repeated spaces** and trim the ends.
5. **Abbreviate street suffixes** to the USPS short form: STREET to ST, AVENUE to AVE, DRIVE to DR, and so on.
6. **Abbreviate directions** like NORTH to N and WEST to W.
7. **Leave unit numbers alone**, so `1007 ALICE ST B` stays a different premises from `1007 ALICE ST`.

### [REVIEW] What I changed before running it
I tested Claude's plan against real rows before accepting it, and found three problems:

| Claude's plan said | Problem I found in the data | What I changed |
|---|---|---|
| Step 6: change WEST to W everywhere | **West End Ave** is a real street name (`3516 WEST END AVE`), and so are `WEST BEND DR`, `EAST LN` and `SOUTH ST`. Changing them everywhere would rewrite the street name itself. | Only change a direction when it's the **last word**, like `214 19TH AVENUE NORTH` becoming `214 19TH AVE N`. |
| Step 2: cutting at the comma removes the city and ZIP | Some rows have the city and ZIP with **no comma**: `2304 EASTLAND AVE NASHVILLE TN 37206` and `1916 CASTLEMAN DRIVE 37215`. | Also remove a trailing `[city] [TN] 12345` pattern. I made sure it doesn't touch `ANTIOCH PIKE` or `TENNESSEE AVE`, which only match when a 5-digit ZIP follows. |
| No step for bad values | One address is literally `NO ADDRESS FOUND.`, and some have **no house number** (`XAVIER DR`) or are **intersections** (`TRIMBLE RD & LINDAWOOD DR`). | Turn placeholders into missing values, and add a `has_house_number` flag so we can leave out rows that can't be tied to one premises. |

In [3]:
# [CLAUDE] Address cleaning, with my [REVIEW] edits included. Every step is a built-in pandas .str method.
SUFFIXES = {
    'STREET': 'ST', 'AVENUE': 'AVE', 'DRIVE': 'DR', 'ROAD': 'RD', 'LANE': 'LN',
    'COURT': 'CT', 'CIRCLE': 'CIR', 'PLACE': 'PL', 'BOULEVARD': 'BLVD',
    'PARKWAY': 'PKWY', 'HIGHWAY': 'HWY', 'TERRACE': 'TER', 'TRAIL': 'TRL',
}
SUFFIX_PATTERN = r'\b(' + '|'.join(SUFFIXES) + r')\b'          # whole words only

# [REVIEW] city/state/ZIP written without a comma, e.g. "... AVE NASHVILLE TN 37206"
CITY_ZIP_PATTERN = (r'\s+(?:NASHVILLE|ANTIOCH|HERMITAGE|MADISON|OLD HICKORY|GOODLETTSVILLE|BRENTWOOD)?'
                    r'\s*(?:TN|TENNESSEE)?\s*\d{5}$')

trash['clean_address'] = (
    trash['Incident Address']
        .str.upper()
        .str.split(',').str[0]                                   # keep the street part only
        .str.replace('.', '', regex=False)                       # RD. -> RD
        .str.replace('#', ' ', regex=False)                      # #416 -> 416
        .str.replace(r'\s+', ' ', regex=True)                    # collapse spaces
        .str.strip()
        .str.replace(CITY_ZIP_PATTERN, '', regex=True)           # [REVIEW] city/ZIP without a comma
        .str.replace(SUFFIX_PATTERN, lambda m: SUFFIXES[m.group(0)], regex=True)
        .str.replace(r' (NORTH|SOUTH|EAST|WEST)$',               # [REVIEW] last word only
                     lambda m: ' ' + m.group(1)[0], regex=True)
        .str.strip()
        .replace({'': pd.NA, 'NO ADDRESS FOUND': pd.NA})         # [REVIEW] placeholders -> missing
)

# [REVIEW] a premises needs a house number; intersections and bare street names don't qualify
trash['has_house_number'] = trash['clean_address'].str.match(r'^\d', na=False)

print('Distinct addresses, raw:        ', trash['Incident Address'].nunique())
print('Distinct addresses, upper-case: ', trash['Incident Address'].str.upper().nunique())
print('Distinct addresses, cleaned:    ', trash['clean_address'].nunique())
print('Rows without a usable address:  ', (~trash['has_house_number']).sum())

Distinct addresses, raw:         14121
Distinct addresses, upper-case:  13781
Distinct addresses, cleaned:     12349
Rows without a usable address:   69


In [4]:
# [CHECK] Show raw spellings that now share one clean address
merged = (trash.dropna(subset=['clean_address'])
               .groupby('clean_address')['Incident Address']
               .agg(['nunique', 'unique']))
print('Clean addresses that combined 2+ raw spellings:', (merged['nunique'] > 1).sum())

examples = ['1007 ALICE ST', '101 AILEEN CT', '3847 CROSS CREEK RD', '214 19TH AVE N',
            '2304 EASTLAND AVE', '1916 CASTLEMAN DR', '4049 MOSS RD']
merged.loc[merged.index.intersection(examples), 'unique'].to_frame('raw spellings combined')

Clean addresses that combined 2+ raw spellings: 1493


,raw spellings combined
clean_address,
1007 ALICE ST,"[1007 Alice St, Nashville, Tennessee, 37218, 1007 Alice St, Nashville, TN 37218, United States, 1007 ALICE..."
101 AILEEN CT,"[101 Aileen Ct, Nashville, TN 37013, United States, 101 Aileen Ct, Antioch, TN 37013, USA]"
1916 CASTLEMAN DR,"[1916 Castleman Dr, 1916 Castleman Drive 37215, 1916 castleman dr]"
214 19TH AVE N,"[214 19th Ave N, 214 19th Avenue North, Nashville, TN, USA, 214 19th Ave N, Nashville, TN 37203, United St..."
2304 EASTLAND AVE,"[2304 Eastland Ave, 2304 Eastland Ave Nashville Tn 37206, 2304 Eastland Ave, Nashville, TN 37206, United S..."
3847 CROSS CREEK RD,"[3847 Cross Creek Rd, 3847 Cross Creek Road, Nashville, TN, USA, 3847 Cross Creek Rd, Nashville, TN 37215,..."
4049 MOSS RD,[4049 Moss Rd.]


In [5]:
# [CHECK] Did anything slip through? Nothing should still contain a city, state, country or comma.
leftover = trash['clean_address'].str.contains(r'NASHVILLE|UNITED STATES|\bUSA\b|,|\bTN \d|\d{5}$', na=False)
print('Suspicious leftovers:', trash.loc[leftover, 'clean_address'].unique())

# [CHECK] West End Ave must survive unchanged (the [REVIEW] fix for step 6)
print(trash.loc[trash['clean_address'].str.contains('WEST END', na=False), 'clean_address'].unique()[:5])

Suspicious leftovers: []
['3516 WEST END AVE' '112 WEST END PL' '102 WEST END CLOSE'
 '3631 WEST END AVE' '109 WEST END CLOSE']


**Result:** No clean address still contains a city, state, country, ZIP or comma, and West End Ave came through unchanged.

**[REVIEW]** My first version of this check flagged `3712 LAUSANNE DR` and three of its neighbors, because "LAUSANNE" contains the letters U-S-A. The check was wrong, not the cleaning, so I made it match `USA` only as a whole word. Validation code can have bugs too.

In [6]:
# [CHECK] Units: is "1007 ALICE ST B" ever written as "1007B ALICE ST"? If so, those would stay split.
letter_first = trash['clean_address'].str.extract(r'^(\d+)([A-Z]) (.+)$').dropna()
letter_first_as_unit = letter_first[0] + ' ' + letter_first[2] + ' ' + letter_first[1]
both_forms = letter_first_as_unit.isin(set(trash['clean_address'].dropna()))
print('Addresses written like 3804A TULANE CT:', len(letter_first.drop_duplicates()))
print('...that also appear as 3804 TULANE CT A:', both_forms.sum())

Addresses written like 3804A TULANE CT: 53
...that also appear as 3804 TULANE CT A: 9


In [7]:
# [REVIEW] Fix it: move a letter stuck to the house number to the end, as a unit (3804A TULANE CT -> 3804 TULANE CT A)
trash['clean_address'] = trash['clean_address'].str.replace(r'^(\d+)([A-Z]) (.+)$', r'\1 \3 \2', regex=True)
trash['has_house_number'] = trash['clean_address'].str.match(r'^\d', na=False)

print('Letter-stuck house numbers left:', trash['clean_address'].str.match(r'^\d+[A-Z] ', na=False).sum())
print('Distinct addresses, cleaned (final):', trash['clean_address'].nunique())
trash.loc[trash['Incident Address'].str.upper().str.startswith('3804A TULANE', na=False),
          ['Incident Address', 'clean_address']].head(2)

Letter-stuck house numbers left: 0
Distinct addresses, cleaned (final): 12341


,Incident Address,clean_address
6079,"3804A Tulane Ct, Nashville, TN 37215, United States",3804 TULANE CT A


**Result:** 53 addresses had the unit letter stuck to the house number (`3804A TULANE CT`). For 9 of them, the same unit also appears in the normal form (`3804 TULANE CT A`), so one apartment was being counted as two addresses. After the fix, none are left. Overall, **14,121 raw spellings became 12,341 standardized addresses.**

### [REVIEW] Is the pandas version actually faster than a custom function?
The assignment says pandas methods are often more efficient than a custom function. Instead of taking that on trust, I timed the same cleaning three ways on all 20,226 rows.

In [8]:
# [CHECK] Time three ways of doing the same cleaning
import re, time

def clean_one(addr):                      # custom function, written carefully
    if pd.isna(addr):
        return addr
    addr = addr.upper().split(',')[0].replace('.', '').replace('#', ' ')
    addr = re.sub(r'\s+', ' ', addr).strip()
    addr = re.sub(CITY_ZIP_PATTERN, '', addr)
    words = [SUFFIXES.get(w, w) for w in addr.split()]
    if words and words[-1] in ('NORTH', 'SOUTH', 'EAST', 'WEST'):
        words[-1] = words[-1][0]
    return ' '.join(words)

def clean_one_naive(addr):                # custom function, the way it's often first written
    if pd.isna(addr):
        return addr
    addr = addr.upper().split(',')[0].replace('.', '').replace('#', ' ')
    addr = re.sub(r'\s+', ' ', addr).strip()
    addr = re.sub(CITY_ZIP_PATTERN, '', addr)
    for long, short in SUFFIXES.items():  # 13 separate regex passes per row
        addr = re.sub(r'\b' + long + r'\b', short, addr)
    return re.sub(r' (NORTH|SOUTH|EAST|WEST)$', lambda m: ' ' + m.group(1)[0], addr)

def vectorized(s):
    return (s.str.upper().str.split(',').str[0]
             .str.replace('.', '', regex=False).str.replace('#', ' ', regex=False)
             .str.replace(r'\s+', ' ', regex=True).str.strip()
             .str.replace(CITY_ZIP_PATTERN, '', regex=True)
             .str.replace(SUFFIX_PATTERN, lambda m: SUFFIXES[m.group(0)], regex=True)
             .str.replace(r' (NORTH|SOUTH|EAST|WEST)$', lambda m: ' ' + m.group(1)[0], regex=True))

def best_of_5(fn):
    times = []
    for _ in range(5):
        start = time.perf_counter(); result = fn(); times.append(time.perf_counter() - start)
    return min(times) * 1000, result

addr = trash['Incident Address']
results = {name: best_of_5(fn) for name, fn in [
    ('pandas .str methods', lambda: vectorized(addr)),
    ('.apply(custom function)', lambda: addr.apply(clean_one)),
    ('.apply(naive custom function)', lambda: addr.apply(clean_one_naive)),
]}
outputs = [r[1].fillna('<NA>').astype(str) for r in results.values()]
print('All three give identical results:', all((o == outputs[0]).all() for o in outputs))
pd.Series({k: round(v[0], 1) for k, v in results.items()}, name='milliseconds (best of 5)')

All three give identical results: True


pandas .str methods               82.4
.apply(custom function)           71.4
.apply(naive custom function)    274.3
Name: milliseconds (best of 5), dtype: float64

**What I found:** The "pandas is faster" advice is only half true for this step.

* A **carefully written custom function** is about as fast as the pandas `.str` chain. Both take well under a tenth of a second for 20,226 rows. pandas string methods still visit each value one at a time behind the scenes, so they don't get the big speedup that math on numbers does.
* A **naive custom function**, the kind that runs 13 separate regex replacements on every row, is about **3 to 4 times slower**.
* All three give **identical results**.

Your timings will differ a little on another computer, but the pattern holds. **Takeaway:** for string cleaning, the pandas version wins on readability more than speed. The real speed difference shows up in Parts 3 and 4, where `groupby` with `cumcount` and rolling windows replace looping over 11,000 addresses one by one.

---
# Part 2.2: Identify rows that represent missed pickups

### Prompt I gave Claude
> "Now help me decide which rows are missed pickups. Explain your rules in plain English first, with no code yet. Then help me hand-check a small sample."

### Claude's approach (before any code)
1. **By request type.** `Trash - Curbside/Alley Missed Pickup` and `Trash - Backdoor` are missed pickups. `Damage to Property` is not. `Trash Collection Complaint` is a judgment call, so check a sample before deciding.
2. **Must have a usable address**, with a house number. Otherwise it can't count toward repeats at "the same premises."
3. **One missed pickup per address per day.** If the same address is reported twice on the same day, it's one missed pickup reported twice, not two.
4. **Clean up the hauler column.** `Metro` and `METRO` are the same value. Fines only apply to **Red River**, so we need to know each row's hauler.

### [REVIEW] My notes on the plan
* Rule 4: 901 rows have **no hauler at all**. Part 1 suggested filling them from `Trash Route`, but I checked and **those rows have no route either** (only 32 of the 716 blank-hauler missed pickups have one). Instead, I fill the hauler from **other requests at the same clean address**, and only when that address has exactly one hauler on record.
* Rule 3: I wanted to see real examples before accepting this, so the manual check below includes an address reported 4 times on one day.

In [9]:
# [CLAUDE] Rule 1: request types that are missed pickups
MISSED_TYPES = ['Trash - Curbside/Alley Missed Pickup', 'Trash - Backdoor']
trash['is_missed_type'] = trash['Request'].isin(MISSED_TYPES)

# [CLAUDE] Rule 4: one spelling per hauler
trash['hauler'] = trash['Trash Hauler'].str.upper()

# [REVIEW] Fill a blank hauler from other requests at the same address, only if they all agree
known = trash.dropna(subset=['hauler', 'clean_address']).groupby('clean_address')['hauler']
single_hauler = known.first()[known.nunique() == 1]
was_blank = trash['hauler'].isna()
trash['hauler'] = trash['hauler'].fillna(trash['clean_address'].map(single_hauler))
print('Blank haulers before:', was_blank.sum(),
      '| filled from same address:', (was_blank & trash['hauler'].notna()).sum(),
      '| still blank:', trash['hauler'].isna().sum())
trash['hauler'].value_counts(dropna=False)

Blank haulers before: 901 | filled from same address: 542 | still blank: 359


hauler
RED RIVER    14810
METRO         3679
WASTE IND     1378
NaN            359
Name: count, dtype: int64

In [10]:
# [CLAUDE] Rules 2 and 3: usable address, then one missed pickup per address per day
candidates = trash[trash['is_missed_type'] & trash['has_house_number']].copy()
candidates = candidates.sort_values(['clean_address', 'Date Opened', 'Request Number'])
candidates['same_day_repeat'] = candidates.duplicated(['clean_address', 'Date Opened'])

missed = candidates[~candidates['same_day_repeat']].copy()

# [ME] How many rows each rule removes, step by step
funnel = pd.Series({
    'All service requests':                         len(trash),
    'Missed-pickup request types':                  trash['is_missed_type'].sum(),
    '...with a usable address (house number)':      len(candidates),
    '...one per address per day (missed pickups)':  len(missed),
    '...where the hauler is Red River':             (missed['hauler'] == 'RED RIVER').sum(),
}, name='rows')
funnel.to_frame()

,rows
All service requests,20226
Missed-pickup request types,17657
...with a usable address (house number),17609
...one per address per day (missed pickups),17243
...where the hauler is Red River,12946


---
## [CHECK] Manual review of a small subset
The point is to read actual rows with my own eyes, not just trust the counts.

**Check 1: Do 10 random rows labeled "missed pickup" actually describe a missed pickup?**

In [11]:
# [CHECK] 1: random sample of rows labeled as missed pickups (fixed seed so it's repeatable)
missed.sample(10, random_state=42)[['Request', 'Description', 'Incident Address', 'clean_address', 'hauler']]

,Request,Description,Incident Address,clean_address,hauler
13882,Trash - Curbside/Alley Missed Pickup,The entire side of the street was missed,1002 N Graycroft Ave,1002 N GRAYCROFT AVE,RED RIVER
699,Trash - Curbside/Alley Missed Pickup,Missed.,2805 Lera Jones Dr,2805 LERA JONES DR,RED RIVER
10443,Trash - Curbside/Alley Missed Pickup,1 cart,"409 Windsor Ct, Nashville, TN 37211, United States",409 WINDSOR CT,RED RIVER
11664,Trash - Curbside/Alley Missed Pickup,"One side of the street not service, Tuesday missed. Curb 3118 and others.","3118 Moorewood Dr, Nashville, TN 37207, United States",3118 MOOREWOOD DR,RED RIVER
18483,Trash - Curbside/Alley Missed Pickup,Entire street/miss trash/last Friday. Curb,"3653 Pin Hook Rd, Nashville, TN 37013, United States",3653 PIN HOOK RD,RED RIVER
9736,Trash - Curbside/Alley Missed Pickup,MISSED- MISSED THE ENTIRE CUL DE SAC,"3004 COUNTRY MEADOW RD, 37013",3004 COUNTRY MEADOW RD,RED RIVER
11104,Trash - Curbside/Alley Missed Pickup,Missed pickup,"941 Havenhill Dr, Nashville, TN 37217, United States",941 HAVENHILL DR,RED RIVER
12414,Trash - Curbside/Alley Missed Pickup,Missed trash,"576 Whispering Hills Dr, Nashville, TN 37211, United States",576 WHISPERING HILLS DR,RED RIVER
12594,Trash - Curbside/Alley Missed Pickup,Trash/curb missed Thursday.,"642 Devon Dr, Nashville, TN 37220, United States",642 DEVON DR,RED RIVER
8987,Trash - Curbside/Alley Missed Pickup,Customer stated this is the 5th missed pickup. She stated there was a man washing carpets outside but ther...,"1820 Champions Dr, Nashville, TN 37211, United States",1820 CHAMPIONS DR,RED RIVER


**Result:** All 10 rows really are missed pickups, so the request type is a reliable label. Two things stood out:

* **4 of the 10 say a whole street, one side of a street or a cul-de-sac was missed**, but each request is filed under one address. The neighbors only count if they called too, so the real number of missed pickups is probably **higher** than the data shows. Code can't fix that. It's a limitation to report.
* A description like "1 cart" is too short to judge alone. That's why I rely on the request type and not on searching descriptions for keywords.

**Check 2: Should `Trash Collection Complaint` rows that mention "miss" be counted?**

In [12]:
# [CHECK] 2: complaints that mention "miss"
complaints = trash[(trash['Request'] == 'Trash Collection Complaint')
                   & trash['Description'].str.contains('miss', case=False, na=False)]
print('Complaint rows mentioning "miss":', len(complaints), 'of',
      (trash['Request'] == 'Trash Collection Complaint').sum())

# How many of them just repeat a missed-pickup request already filed for that address that day?
already_counted = complaints.set_index(['clean_address', 'Date Opened']).index.isin(
    candidates.set_index(['clean_address', 'Date Opened']).index)
print('...already covered by a missed-pickup request at the same address on the same day:', already_counted.sum())
complaints.sample(10, random_state=7)[['Date Opened', 'Description', 'clean_address']]

Complaint rows mentioning "miss": 691 of 2312
...already covered by a missed-pickup request at the same address on the same day: 36


,Date Opened,Description,clean_address
13140,2019-05-30,MISSED...ENTIRE COMPLEX OF APPX 42 HOMES...GLOSTER SQUARE. RESIDENT SAYS IT IS RIDICULOUS THAT EACH INDIVI...,2138 GOLF CLUB LN
9293,2019-02-01,Caller said she has been missed every friday for a month. They trash truck has already come this morning ...,3004 FIELDSTONE DR
5792,2018-08-29,Missed every other week customer tired of calling the Hub to make driver come back to get trash 3 days lat...,5105 BUENA VISTA PIKE
93,2017-11-03,missed - everyone except his picked up,641 TRUXTON CT
15298,2019-07-22,"Whole circle missed, & Longhunter Ct. Trash missed last Friday.",209 LONGHUNTER CIR
17226,2019-08-21,RESIDENT WANTS TO KNOW WHY HIS ADDRESS IS CONSISTENTLY MISSED. THIS IS HAPPENING FREQUENTLY.,1501 RIVERSIDE DR
20096,2019-10-30,today makes 3 weeks missed trash/curb {sit's on 40th Ave N} side. {Wednesday}\ndoes not have access to all...,4001 NEBRASKA AVE
564,2017-12-07,cust says pickup was missed for him 3 weeks in a row,5513 MORROW RD
18408,2019-09-20,"Has Backdoor/Trash pickup, 2 weeks of being missed, elderly and if set any closer of where she places her ...",4285 LAVERGNE COUCHVILLE PIKE
19547,2019-10-17,**MISSED AGAIN YESTERDAY****SUPERVISOR DID NOT MAKE CONTACT AS REQUESTED***PLEASE CALL THIS TIME********SI...,2813 DELAWARE AVE


**Decision: leave complaints out.** Many of these rows are about missed trash, but most describe a **pattern** ("missed every friday for a month", "3 weeks in a row", "consistently missed") rather than one missed pickup on the date the request was opened. Only 36 of the 691 fall on the same address and day as a missed-pickup request, and those would be duplicates anyway. The other 655 may point to extra misses, but there's no way to tell which pickup date each one means. Counting each as one missed pickup would be a guess, so the Part 3 totals are **conservative** (on the low side).

**Check 3: Does the same-day rule do the right thing?** Here is `4049 MOSS RD` with every raw row.

In [13]:
# [CHECK] 3: one address, every raw row, side by side with what the rules kept
cols = ['Request Number', 'Date Opened', 'Request', 'Incident Address', 'Description', 'hauler']
moss = candidates.loc[candidates['clean_address'] == '4049 MOSS RD', cols + ['same_day_repeat']]
moss

,Request Number,Date Opened,Request,Incident Address,Description,hauler,same_day_repeat
320,28848,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,False
321,28849,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True
322,28850,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True
323,28851,2017-11-17,Trash - Curbside/Alley Missed Pickup,4049 Moss Rd.,Had trash at road last night but still missed pickup this morning,NaN,True


**Result:** The rule works as intended. These are four rows with **consecutive request numbers (28848 to 28851)**, the same date and an identical description. That is one phone call entered four times. The rule keeps the first row and marks the other three as `same_day_repeat`. Without it, this address would generate **$600 in fines for a single missed pickup**.

The hauler is also blank here, and no other request at this address names one, so this pickup can't be credited to Red River. It's one of the **245 missed pickups that still have no hauler** after the fill step.

**Check 4: The address with the most missed pickups, counted by hand.**

In [14]:
# [CHECK] 4: the busiest address after cleaning. Count its rows by hand from the raw data.
top_address = missed['clean_address'].value_counts().idxmax()
print('Busiest address:', top_address, '| missed pickups counted by code:',
      missed['clean_address'].value_counts().max())

# Every raw row for this address, all request types, so I can count the missed pickups myself
raw_rows = trash.loc[trash['clean_address'] == top_address, cols].sort_values('Date Opened')
print('All requests at this address, by type:')
print(raw_rows['Request'].value_counts().to_string())
print('Distinct raw spellings:', raw_rows['Incident Address'].nunique())
print('Haulers on its missed pickups:', missed.loc[missed['clean_address'] == top_address, 'hauler'].value_counts().to_dict())
print("Red River's busiest addresses:",
      missed.loc[missed['hauler'] == 'RED RIVER', 'clean_address'].value_counts().head(3).to_dict())
raw_rows

Busiest address: 110 GEORGE L DAVIS BLVD | missed pickups counted by code: 25
All requests at this address, by type:
Request
Trash - Curbside/Alley Missed Pickup    25
Trash Collection Complaint               4
Distinct raw spellings: 5
Haulers on its missed pickups: {'METRO': 25}
Red River's busiest addresses: {'12546 OLD HICKORY BLVD': 21, '1584 BELL RD': 19, '5135 HICKORY HOLLOW PKWY': 18}


,Request Number,Date Opened,Request,Incident Address,Description,hauler
989,37104,2018-01-05,Trash Collection Complaint,110 GEORGE L DAVIS BLVD,missed pick up again & again,METRO
2335,53873,2018-03-15,Trash Collection Complaint,110 George L Davis Blvd,missed all the time - said no alley access,METRO
2496,55760,2018-03-23,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,missed all the time - said no alley access {carts are placed where they always been for 6 years. Has to ke...,METRO
3793,71125,2018-05-30,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,trash is full needs pick up. Matt stated that the pickup has been all days of the week and never on the co...,METRO
3977,73434,2018-06-07,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Caller wants a call ba...,METRO
4260,77172,2018-06-21,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,trash is full needs pick up. Matt stated that the pickup has been all days of the week and never on the co...,METRO
4798,82908,2018-07-13,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Thursday's pickup. {Cu...,METRO
5853,94313,2018-08-31,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,Missed. Caller states that this is a business and they have been missed every week. Thursday's pickup. {Cu...,METRO
7787,122089,2018-12-07,Trash - Curbside/Alley Missed Pickup,110 George L Davis Blvd,missed carts,METRO
7813,122323,2018-12-10,Trash Collection Complaint,110 George L Davis Blvd,Missed again- he has to call about every week. Wanted to speak with supervisor. Transferred to Josh.,METRO


**Hand count:** I counted the `Trash - Curbside/Alley Missed Pickup` rows in the table above and got **25**, which matches the code. The 4 complaint rows are correctly left out. This address teaches two lessons:

* **Cleaning mattered.** The 25 missed pickups were spread across **5 different spellings**, and the most common single spelling had only 12. Without standardizing, this address would look about half as bad as it is.
* **The busiest address is Metro's, not Red River's.** All 25 rows have hauler `METRO`, so none of them should become a Red River fine. If I had skipped the hauler filter, the address with the most fines in my report would be one Red River isn't responsible for. Red River's busiest addresses are listed in the output above.

One related row, `110 george l davis blvd E`, stays separate because a trailing letter is treated as a unit (Part 1 assumption).

**Check 5: Nothing that isn't a missed pickup got in.**

In [15]:
# [CHECK] 5: only the two missed-pickup request types, no blank addresses, no same-day duplicates
print('Request types in result:', missed['Request'].unique().tolist())
print('Blank clean addresses:  ', missed['clean_address'].isna().sum())
print('Same-day duplicates:    ', missed.duplicated(['clean_address', 'Date Opened']).sum())
print('Damage rows included:   ', (missed['Request'] == 'Damage to Property').sum())

Request types in result: ['Trash - Curbside/Alley Missed Pickup', 'Trash - Backdoor']
Blank clean addresses:   0
Same-day duplicates:     0
Damage rows included:    0


---
## Part 2 summary
Part 2 turns **20,226 raw service requests** into **17,243 missed pickups at 11,133 standardized addresses**. Of those, **12,946 missed pickups at 8,246 addresses belong to Red River**.

| Step | What it does | Evidence it works |
|---|---|---|
| Address cleaning | 14,121 raw spellings become 12,341 addresses | 1,493 addresses combined from 2+ spellings; no city or ZIP left over; West End Ave untouched |
| Missed-pickup types | Curbside/Alley + Backdoor only | 10-row sample all real misses; complaints checked and left out on purpose |
| Usable address | Drops 48 missed-pickup rows with no house number | Intersections and bare street names can't be one premises |
| One per address per day | Removes 366 repeat reports | 4049 Moss Rd: 4 entries of one call become 1 |
| Hauler | `Metro` = `METRO`; 542 blanks filled from the same address | The busiest address (25 misses) is Metro's and won't be fined |

**Limitations carried into Part 3:**

* 245 missed pickups still have no hauler and can't be credited to Red River.
* Whole-street misses are under-reported.
* Complaints that may describe extra misses are left out.

All three push the totals **down**, so the fines in Part 3 are a floor, not a ceiling.

**Next (Part 3):** number each Red River address's missed pickups in date order with `groupby().cumcount()`. The first is $0 and every later one is $200.

---
# Part 3: Calculating fines (base rule)
**Rule:** the first missed pickup at an address costs $0. **Every** missed pickup after that costs $200.

### Prompt I gave Claude
> "Using the `missed` dataframe from Part 2, suggest an approach for calculating the fines. Explain it in plain English first, with no code. I want built-in pandas methods, not a loop over addresses or a custom function in `.apply()`."

### Claude's approach (before any code)
1. **Keep only Red River's missed pickups.** The fines come from Metro's contract with Red River.
2. **Sort by address, then by date.** "First" only means something once the rows are in time order.
3. **Number the pickups within each address** with `groupby('clean_address').cumcount()`. This gives 0 for the first pickup, 1 for the second, and so on. It's one vectorized step, with no loop.
4. **Set the fine:** $200 where the number is 1 or more, otherwise $0.
5. **Report** total missed pickups (rows), fined pickups (rows with a fine) and damages (sum of fines).

### [REVIEW] My notes before running it
* **Filter to Red River first, then number. The order matters.** A few addresses have more than one hauler in their history. If I numbered first, a Metro miss could make Red River's first miss look like a "second" and fine Red River $200 for it. Filtering first means Red River's own first miss is always free.
* **Ties on the same date.** Part 2 already keeps one missed pickup per address per day, so two rows can't tie. I still sort by `Request Number` as a last tiebreaker, so the order never depends on how the file happened to be sorted.
* **A shortcut to check the answer.** Every address gets exactly one free pickup, so **fined pickups = missed pickups − number of addresses**. That gives me an independent check that doesn't use `cumcount` at all.

In [16]:
# [CLAUDE] Steps 1-2: Red River only, in time order within each address
red_river = (missed[missed['hauler'] == 'RED RIVER']
             .sort_values(['clean_address', 'Date Opened', 'Request Number'])
             .copy())

# [CLAUDE] Step 3: number each address's missed pickups (0 = first, 1 = second, ...)
red_river['pickup_number'] = red_river.groupby('clean_address').cumcount()

# [CLAUDE] Step 4: $0 for the first, $200 for every one after
red_river['fine'] = (red_river['pickup_number'] >= 1) * 200

red_river[['clean_address', 'Date Opened', 'Request', 'pickup_number', 'fine']].head(8)

,clean_address,Date Opened,Request,pickup_number,fine
10929,100 BLUEFIELD SQUARE,2019-03-15,Trash - Curbside/Alley Missed Pickup,0,0
2214,100 BRAXTON HILL CT,2018-03-07,Trash - Curbside/Alley Missed Pickup,0,0
7727,100 BROOK HOLLOW RD,2018-12-05,Trash - Curbside/Alley Missed Pickup,0,0
421,100 CEDARMONT CT,2017-11-28,Trash - Curbside/Alley Missed Pickup,0,0
9841,100 CLYDELAN CT,2019-02-14,Trash - Curbside/Alley Missed Pickup,0,0
20162,100 DESOTO DR,2019-10-31,Trash - Curbside/Alley Missed Pickup,0,0
20079,100 DYER CT,2019-10-29,Trash - Curbside/Alley Missed Pickup,0,0
7794,100 GRAEME DR,2018-12-07,Trash - Curbside/Alley Missed Pickup,0,0


In [17]:
# [CLAUDE] Step 5: the three numbers the assignment asks for
total_missed  = len(red_river)
total_fined   = (red_river['fine'] > 0).sum()
total_damages = red_river['fine'].sum()

report = pd.DataFrame({
    'Result': ['Total missed pickups (Red River)', 'Total fined pickups', 'Total damages'],
    'Value':  [f'{total_missed:,}', f'{total_fined:,}', f'${total_damages:,.0f}'],
}).set_index('Result')
report

,Value
Result,
Total missed pickups (Red River),"12,946"
Total fined pickups,"4,700"
Total damages,"$940,000"


## [CHECK] Validation
**Check 1: Count the busiest Red River address by hand.**

In [18]:
# [CHECK] 1: one address, every missed pickup, in order
check_address = red_river['clean_address'].value_counts().idxmax()
one = red_river.loc[red_river['clean_address'] == check_address,
                    ['Request Number', 'Date Opened', 'Incident Address', 'pickup_number', 'fine']]
print(check_address, '|', len(one), 'missed pickups | fines by code:', f"${one['fine'].sum():,}")
print('By hand: first one is free, so', len(one) - 1, 'fined x $200 =', f"${(len(one) - 1) * 200:,}")
one

12546 OLD HICKORY BLVD | 21 missed pickups | fines by code: $4,000
By hand: first one is free, so 20 fined x $200 = $4,000


,Request Number,Date Opened,Incident Address,pickup_number,fine
5582,91312,2018-08-17,12546 Old Hickory Blvd,0,0
5878,94453,2018-09-01,"12546 Old Hickory Blvd, Antioch, TN, USA",1,200
6324,101976,2018-09-21,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",2,200
6610,106847,2018-10-05,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",3,200
7052,112505,2018-10-26,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",4,200
7424,117479,2018-11-17,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",5,200
7796,122151,2018-12-07,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",6,200
8160,125333,2018-12-21,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",7,200
8283,126544,2018-12-29,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",8,200
8853,132925,2019-01-18,"12546 Old Hickory Blvd, Nashville, TN 37013, United States",9,200


**Hand count:** I counted 21 rows. The first one (Aug 17, 2018) is free, and the other 20 are $200 each: **20 × $200 = $4,000, which matches the code.** `pickup_number` runs from 0 to 20 with no gaps, and the dates are in order.

This address shows up under **3 different spellings**. Without the Part 2 cleaning, two of these 21 would each look like a separate "first" miss, and the fine would drop from $4,000 to $3,600.

**Check 2: The shortcut formula and a second method should give the same totals.**

In [19]:
# [CHECK] 2a: shortcut from the [REVIEW] note: fined = missed - addresses
addresses = red_river['clean_address'].nunique()
print('Missed pickups - addresses =', total_missed, '-', addresses, '=', total_missed - addresses,
      '| cumcount method:', total_fined, '| match:', total_missed - addresses == total_fined)

# [CHECK] 2b: a completely different method. Count per address, then (count - 1) x $200.
per_address = red_river['clean_address'].value_counts()
other_way = ((per_address - 1) * 200).sum()
print('value_counts method damages:', f'${other_way:,}', '| match:', other_way == total_damages)

Missed pickups - addresses = 12946 - 8246 = 4700 | cumcount method: 4700 | match: True
value_counts method damages: $940,000 | match: True


**Check 3: Edge cases. An address with exactly one missed pickup must owe $0, and one with exactly two must owe $200.**

In [20]:
# [CHECK] 3: smallest cases
one_miss  = per_address[per_address == 1].index[0]
two_miss  = per_address[per_address == 2].index[0]
for addr in [one_miss, two_miss]:
    rows = red_river.loc[red_river['clean_address'] == addr, ['clean_address', 'Date Opened', 'pickup_number', 'fine']]
    print(rows.to_string(index=False), '\n')

print('Addresses with 1 missed pickup (owe $0):', (per_address == 1).sum())
print('Every first pickup has fine 0:', (red_river.loc[red_river['pickup_number'] == 0, 'fine'] == 0).all())
print('Every later pickup has fine 200:', (red_river.loc[red_river['pickup_number'] >= 1, 'fine'] == 200).all())

       clean_address Date Opened  pickup_number  fine
100 BLUEFIELD SQUARE  2019-03-15              0     0 

      clean_address Date Opened  pickup_number  fine
100 NASHBORO GREENS  2019-07-01              0     0
100 NASHBORO GREENS  2019-07-23              1   200 

Addresses with 1 missed pickup (owe $0): 5874
Every first pickup has fine 0: True
Every later pickup has fine 200: True


**Check 4: How much does each Part 1 and 2 decision change the answer?** I re-ran the base rule while undoing one decision at a time. This shows which assumptions matter most, so a reader can judge them.

In [21]:
# [CHECK] 4: sensitivity, the same base rule with one decision undone at a time
def base_rule(rows, address_col='clean_address'):
    counts = rows[address_col].value_counts()
    fined = int((counts - 1).sum())
    return pd.Series({'missed pickups': len(rows), 'fined pickups': fined, 'damages': fined * 200})

usable = candidates                                        # missed types + house number, before same-day rule
scenarios = pd.DataFrame({
    'My answer (all rules)':               base_rule(red_river),
    'No same-day duplicate rule':          base_rule(usable[usable['hauler'] == 'RED RIVER']),
    'No address cleaning (raw text)':      base_rule(red_river, address_col='Incident Address'),
    'All haulers, not just Red River':     base_rule(missed),
    'Blank haulers not filled in':         base_rule(missed[missed['Trash Hauler'].str.upper() == 'RED RIVER']),
}).T
scenarios['damages vs mine'] = scenarios['damages'] - scenarios.loc['My answer (all rules)', 'damages']
signed_dollars = lambda v: f'+${v:,.0f}' if v > 0 else (f'-${-v:,.0f}' if v < 0 else '$0')
scenarios.style.format('{:,.0f}', subset=['missed pickups', 'fined pickups']) \
               .format('${:,.0f}', subset=['damages']).format(signed_dollars, subset=['damages vs mine'])

,missed pickups,fined pickups,damages,damages vs mine
My answer (all rules),"12,946","4,700","$940,000",$0
No same-day duplicate rule,"13,231","4,985","$997,000","+$57,000"
No address cleaning (raw text),"12,946","3,666","$733,200","-$206,800"
"All haulers, not just Red River","17,243","6,110","$1,222,000","+$282,000"
Blank haulers not filled in,"12,642","4,406","$881,200","-$58,800"


**What the table shows:**

* **Address cleaning matters most.** Without it, damages fall by **$206,800 (about 22%)**. There are the same 12,946 missed pickups, but repeat misses at one house hide behind different spellings, and each spelling gets its own free "first" miss.
* **The Red River filter keeps out $282,000** in fines for misses by Metro and Waste Industries, which aren't covered by this contract.
* **The same-day rule removes $57,000** that came only from the same miss being reported more than once.
* **Filling in blank haulers adds $58,800.** This is the one place I went beyond what the raw data says. If a reader rejects that step, the answer is **$881,200**, so I report it as a range.

All five versions use the same `base_rule`, which counts per address and charges (count − 1) × $200. That means the differences come only from the data decisions, not from the fine logic.

In [22]:
# [ME] Save the fines by address for the submission
import os
os.makedirs('results', exist_ok=True)
fines_by_address = (red_river.assign(is_fined=red_river['fine'] > 0)
                              .groupby('clean_address')
                              .agg(missed_pickups=('fine', 'size'),
                                   fined_pickups=('is_fined', 'sum'),
                                   damages=('fine', 'sum'),
                                   first_miss=('Date Opened', 'min'),
                                   last_miss=('Date Opened', 'max'))
                              .sort_values(['damages', 'missed_pickups'], ascending=False))
fines_by_address.to_csv('results/red_river_fines_by_address.csv')
print(len(fines_by_address), 'addresses saved | damages add up:', fines_by_address['damages'].sum() == total_damages)
fines_by_address.head(10)

8246 addresses saved | damages add up: True


,missed_pickups,fined_pickups,damages,first_miss,last_miss
clean_address,,,,,
12546 OLD HICKORY BLVD,21,20,4000,2018-08-17,2019-06-11
1584 BELL RD,19,18,3600,2017-11-27,2019-08-30
5135 HICKORY HOLLOW PKWY,18,17,3400,2018-07-16,2019-06-18
3710 N NATCHEZ CT,16,15,3000,2017-11-29,2019-03-21
607 ESTES RD,16,15,3000,2019-02-01,2019-10-23
802 CRESCENT RD,16,15,3000,2019-04-03,2019-10-25
1537 HARDING PL,15,14,2800,2018-03-08,2019-10-16
320 OLD HICKORY BLVD,15,14,2800,2018-01-22,2019-10-02
206 ROLLING FORK CT,14,13,2600,2018-08-23,2019-10-10


---
## Part 3 report (base rule)
| Result | Value |
|---|---:|
| **Total missed pickups (Red River)** | **12,946** |
| **Total fined pickups** | **4,700** |
| **Total damages** | **$940,000** |

**How to read it:** Red River had missed pickups at **8,246 addresses**. **5,874 of them (71%) were missed only once** and owe nothing. The 4,700 fines come from the other **2,372 addresses** that were missed more than once. The worst, 12546 Old Hickory Blvd, was missed 21 times and owes **$4,000**.

**Why I trust it:**

* A hand count of one address matches the code.
* Two independent methods give exactly the same totals: `cumcount`, and `value_counts` with (count − 1).
* The edge cases are right: 1 miss costs $0 and 2 misses cost $200.
* Every first pickup is $0 and every later one is $200.

**Range:** the answer is **$881,200 to $940,000**, depending on whether blank haulers are filled in from the same address. It is still a **floor**, for three reasons:

* Complaints that may describe extra misses are left out.
* Whole-street misses are only counted at the address that called.
* 245 missed pickups have no hauler at all.

If "total missed pickups" is meant to include every hauler, the number is 17,243. The fines are Red River's alone either way.

**Saved:** [`results/red_river_fines_by_address.csv`](results/red_river_fines_by_address.csv) has one row per Red River address, with its missed pickups, fines and dates.

---
# Part 4: Challenge rule ($500 inside a 6-month window)
**Contract rule:** a missed pickup is fined **$500** if it is the **third or later** missed pickup at the same address **within a 6-month rolling window**.

**How I'm reading it:** each missed pickup gets **one** fine, the highest one that applies.

| A missed pickup is... | Fine |
|---|---:|
| The 3rd or later miss at that address in the 6 months up to and including that day | **$500** |
| Not the address's first miss ever, but not 3rd-in-6-months either | **$200** |
| The address's very first missed pickup | **$0** |

**"6 months"** means calendar months. A pickup on Aug 15, 2019 looks back to Feb 15, 2019, and both ends count. Check 5 shows how much the answer moves under other reasonable definitions, like 182 or 183 days.

### Prompt I gave Claude
> "Give me several possible approaches for flagging a missed pickup that is the 3rd or later at the same address within a 6-month rolling window. Rank them by efficiency, and explain each in plain English with no code. The data is `red_river`, already sorted by address and date."

### Claude's approaches (Claude's own ranking, slowest to fastest)
| # | Approach | How it works | Problem |
|---|---|---|---|
| 1 | **Loop over every row** (`iterrows`) | For each missed pickup, filter the whole table to the same address and the past 6 months, then count the rows | Scans all 12,946 rows **for every row**, about 168 million comparisons. Very slow. |
| 2 | **`groupby().apply(custom function)`** | Loop through each address's dates inside a Python function | Only loops inside each address, so it's much faster than #1, but it's still a custom function. |
| 3 | **Self-merge** | Join the table to itself on address, keep pairs less than 6 months apart, count per pickup | Vectorized, but it builds every pair of misses at an address, so memory grows with the **square** of the misses per address. |
| 4 | **`groupby().rolling('182D')`** | pandas' built-in time window counts the misses in the last 182 days for each row | Fast and built for this, but windows must be a **fixed number of days** (no calendar months), and the result comes back with a different index to line back up. |

### [REVIEW] Guiding Claude to a better answer
Claude ranked #4, the rolling window, as best. I asked a follow-up: *"We don't need the full count in the window, only whether it's at least 3. Is there a shortcut?"* That led to a fifth approach:

| # | Approach | How it works |
|---|---|---|
| 5 | **`groupby().shift(2)`** | The rows are already in date order per address. A pickup is the 3rd or later in its window **exactly when the pickup two before it, at the same address, is still inside the window**. If the 2nd-previous miss is in the window, the previous one must be too, because it's even more recent. So it takes one `shift(2)` and one date comparison. |

### Which one I chose, and why
**I'm using #5, `shift(2)`, and #4, `rolling`, as an independent check.**

* **Fastest:** one vectorized shift and one comparison, with no counting and no realigning.
* **Exact calendar months:** it compares against `date − 6 months` (`pd.DateOffset(months=6)`), which `rolling` can't do.
* **Easiest to read and explain:** "were there already 2 misses in the last 6 months?"
* **Easy to check:** the 182-day version of #5 must match `rolling('182D')` row for row. A plain loop (#2) gives a third, independent answer to compare against.

In [23]:
# [CLAUDE] Approach 5: is the pickup two back at the same address still inside the 6-month window?
import numpy as np

red_river = red_river.sort_values(['clean_address', 'Date Opened', 'Request Number'])
red_river['two_back_date'] = red_river.groupby('clean_address')['Date Opened'].shift(2)    # NaT for 1st and 2nd
red_river['window_start']  = red_river['Date Opened'] - pd.DateOffset(months=6)
red_river['third_in_6_months'] = red_river['two_back_date'] >= red_river['window_start']    # NaT compares as False

# One fine per pickup, the highest that applies
red_river['challenge_fine'] = np.select(
    [red_river['third_in_6_months'], red_river['pickup_number'] >= 1],
    [500, 200],
    default=0,
)

red_river.loc[red_river['clean_address'] == '1584 BELL RD',
              ['clean_address', 'Date Opened', 'pickup_number', 'two_back_date', 'window_start', 'challenge_fine']].head(8)

,clean_address,Date Opened,pickup_number,two_back_date,window_start,challenge_fine
396,1584 BELL RD,2017-11-27,0,NaT,2017-05-27,0
1650,1584 BELL RD,2018-02-09,1,NaT,2017-08-09,200
1887,1584 BELL RD,2018-02-22,2,2017-11-27,2017-08-22,500
2608,1584 BELL RD,2018-03-30,3,2018-02-09,2017-09-30,500
2913,1584 BELL RD,2018-04-12,4,2018-02-22,2017-10-12,500
2935,1584 BELL RD,2018-04-13,5,2018-03-30,2017-10-13,500
3086,1584 BELL RD,2018-04-20,6,2018-04-12,2017-10-20,500
3129,1584 BELL RD,2018-04-23,7,2018-04-13,2017-10-23,500


In [24]:
# [CLAUDE] The challenge-rule report, next to the base rule from Part 3
fine_counts = red_river['challenge_fine'].value_counts().reindex([0, 200, 500])
challenge_total = red_river['challenge_fine'].sum()

challenge_report = pd.DataFrame({
    'Base rule (Part 3)': [f'{total_missed:,}', f'{(red_river["fine"] == 0).sum():,}', f'{total_fined:,}', '0',
                           f'${total_damages:,.0f}'],
    'Challenge rule':     [f'{len(red_river):,}', f'{fine_counts[0]:,}', f'{fine_counts[200]:,}', f'{fine_counts[500]:,}',
                           f'${challenge_total:,.0f}'],
}, index=['Missed pickups', '$0 (first miss)', '$200 fines', '$500 fines', 'Total damages'])
print('Extra damages from the challenge rule:', f'${challenge_total - total_damages:,.0f}')
challenge_report

Extra damages from the challenge rule: $479,700


,Base rule (Part 3),Challenge rule
Missed pickups,"12,946","12,946"
$0 (first miss),"8,246","8,246"
$200 fines,"4,700","3,101"
$500 fines,0,"1,599"
Total damages,"$940,000","$1,419,700"


## [CHECK] Checks on the solution
**Check 1: The `rolling` window (approach 4) must agree with `shift(2)` (approach 5) row by row.** `rolling` only does fixed-length windows, so both use 182 days here. For both, the window is the 182 days before a pickup, including that day.

In [25]:
# [CHECK] 1: approach 4 (rolling count) vs approach 5 (shift) with the same 182-day window
rolling_count = (red_river.set_index('Date Opened')
                          .groupby('clean_address')['pickup_number']
                          .rolling('182D').count()
                          .rename('misses_in_last_182_days')
                          .reset_index())                       # keyed by (address, date): unique after Part 2
compare = red_river.merge(rolling_count, on=['clean_address', 'Date Opened'], how='left', validate='one_to_one')

by_rolling = compare['misses_in_last_182_days'] >= 3
by_shift   = compare['two_back_date'] > compare['Date Opened'] - pd.Timedelta(days=182)
print('Rows compared:', len(compare), '| flagged by rolling:', by_rolling.sum(),
      '| flagged by shift:', by_shift.sum(), '| identical on every row:', (by_rolling == by_shift).all())

Rows compared: 12946 | flagged by rolling: 1589 | flagged by shift: 1589 | identical on every row: True


**Check 2: A plain, obviously correct loop (approach 2) must give the same $500 flags with the real calendar-month window.** It's slow and written for clarity, not speed, and it shares no code with `shift(2)`.

In [26]:
# [CHECK] 2: brute force, counting by hand in Python for every address
from dateutil.relativedelta import relativedelta

def flag_by_counting(dates):
    """For each date, count misses from (date - 6 months) through date. 3 or more -> True."""
    flags = []
    for i, day in enumerate(dates):
        start = day - relativedelta(months=6)
        in_window = sum(1 for earlier in dates[:i + 1] if earlier >= start)
        flags.append(in_window >= 3)
    return flags

brute = []
for address, rows in red_river.groupby('clean_address', sort=False):
    brute.extend(zip(rows.index, flag_by_counting(list(rows['Date Opened']))))
brute = pd.Series(dict(brute))

print('Brute-force $500 flags:', brute.sum(), '| shift(2) $500 flags:', red_river['third_in_6_months'].sum(),
      '| identical on every row:', (brute.reindex(red_river.index) == red_river['third_in_6_months']).all())

Brute-force $500 flags: 1599 | shift(2) $500 flags: 1599 | identical on every row: True


**Check 3: Explain one address by hand.**

In [27]:
# [CHECK] 3: an address that has all three fine levels
levels = red_river.groupby('clean_address')['challenge_fine'].agg(['nunique', 'size'])
example = levels[(levels['nunique'] == 3) & levels['size'].between(6, 9)].index[0]

walk = red_river.loc[red_river['clean_address'] == example,
                     ['Date Opened', 'pickup_number', 'two_back_date', 'window_start', 'challenge_fine']].copy()
walk['days_since_two_back'] = (walk['Date Opened'] - walk['two_back_date']).dt.days
print(example, '| total under challenge rule:', f"${walk['challenge_fine'].sum():,}",
      '| under base rule:', f"${(len(walk) - 1) * 200:,}")
walk

1005 NOELTON AVE | total under challenge rule: $2,100 | under base rule: $1,200


,Date Opened,pickup_number,two_back_date,window_start,challenge_fine,days_since_two_back
1605,2018-02-08,0,NaT,2017-08-08,0,NaN
1674,2018-02-12,1,NaT,2017-08-12,200,NaN
3301,2018-05-03,2,2018-02-08,2017-11-03,500,84.0
3417,2018-05-09,3,2018-02-12,2017-11-09,500,86.0
5197,2018-08-02,4,2018-05-03,2018-02-02,500,91.0
14366,2019-06-27,5,2018-05-09,2018-12-27,200,414.0
20157,2019-10-31,6,2018-08-02,2019-04-30,200,455.0


**By hand, for 1005 Noelton Ave:**

| Miss | Misses in the 6 months up to that day | Fine | Why |
|---|---|---:|---|
| Feb 8, 2018 | 1 | $0 | First miss ever |
| Feb 12, 2018 | 2 | $200 | Not first, but only 2 in the window |
| May 3, 2018 | 3 (Feb 8, Feb 12, May 3) | **$500** | 3rd within 6 months |
| May 9, 2018 | 4 | **$500** | Feb 12 is still inside the window (starts Nov 9, 2017) |
| Aug 2, 2018 | 4 (Feb 8 through Aug 2; window starts Feb 2) | **$500** | Still 3 or more in the window |
| Jun 27, 2019 | 1 | $200 | Over a year since the last miss, so the window is empty. Not first ever, so $200. |
| Oct 31, 2019 | 2 (Jun 27, Oct 31) | $200 | Only 2 in the window |

$0 + $200 + 3 × $500 + 2 × $200 = **$2,100**, which matches the code. The base rule would charge 6 × $200 = $1,200. This address also shows the window **rolling forward**: after a quiet year, the fines drop back to $200 instead of staying at $500.

**Check 4: Rules that must always hold.**

In [28]:
# [CHECK] 4: invariants
first_rows = red_river['pickup_number'] == 0
print('Every first miss is $0:                     ', (red_river.loc[first_rows, 'challenge_fine'] == 0).all())
print('$500 only on the 3rd miss or later:          ', (red_river.loc[red_river['challenge_fine'] == 500, 'pickup_number'] >= 2).all())
print('No address owes less than under the base rule:',
      (red_river.groupby('clean_address')['challenge_fine'].sum()
       >= red_river.groupby('clean_address')['fine'].sum()).all())
print('Each $500 = a $200 fine raised by $300:       ',
      challenge_total - total_damages == 300 * fine_counts[500])
misses_at_address = red_river.groupby('clean_address')['pickup_number'].transform('size')
small = red_river[misses_at_address <= 2]
print('Addresses with 2 or fewer misses unchanged:   ', (small['challenge_fine'] == small['fine']).all())

Every first miss is $0:                      True
$500 only on the 3rd miss or later:           True
No address owes less than under the base rule: True
Each $500 = a $200 fine raised by $300:        True
Addresses with 2 or fewer misses unchanged:    True


**Check 5: How much does the definition of "6 months" matter?**

In [29]:
# [CHECK] 5: sensitivity to the window definition
def challenge_total_for(in_window):
    return int(np.select([in_window, red_river['pickup_number'] >= 1], [500, 200], default=0).sum())

two_back, day = red_river['two_back_date'], red_river['Date Opened']
definitions = {
    '6 calendar months, ends included (my choice)': two_back >= day - pd.DateOffset(months=6),
    '6 calendar months, start excluded':            two_back >  day - pd.DateOffset(months=6),
    '180 days':                                     two_back >  day - pd.Timedelta(days=180),
    '182 days (= rolling 182D)':                    two_back >  day - pd.Timedelta(days=182),
    '183 days':                                     two_back >  day - pd.Timedelta(days=183),
}
sensitivity = pd.DataFrame({
    '$500 fines': {k: int(v.sum()) for k, v in definitions.items()},
    'total damages': {k: challenge_total_for(v) for k, v in definitions.items()},
})
sensitivity['vs my choice'] = sensitivity['total damages'] - sensitivity['total damages'].iloc[0]
sensitivity.style.format({'$500 fines': '{:,}', 'total damages': '${:,.0f}', 'vs my choice': '{:+,.0f}'})

,$500 fines,total damages,vs my choice
"6 calendar months, ends included (my choice)","1,599","$1,419,700",+0
"6 calendar months, start excluded","1,593","$1,417,900","-1,800"
180 days,"1,582","$1,414,600","-5,100"
182 days (= rolling 182D),"1,589","$1,416,700","-3,000"
183 days,"1,601","$1,420,300",+600


**Result:** The definition barely matters. Across five reasonable readings of "6 months," the total moves by at most **$5,100**, which is under 0.4%. Calendar months with both ends included is the most natural reading of the contract, so that's my answer. The others are within rounding.

### [CHECK] Efficiency: timing the approaches

In [30]:
# [CHECK] Time approaches 1-5 on the same data
import time
dates = red_river['Date Opened']
addresses = red_river['clean_address']

def approach_1_loop_rows(rows):                       # iterrows + filter the whole table each time
    out = []
    for _, r in rows.iterrows():
        same = red_river[(addresses == r['clean_address'])
                         & (dates <= r['Date Opened']) & (dates >= r['window_start'])]
        out.append(len(same) >= 3)
    return out

def approach_2_groupby_loop():
    return [f for _, rows in red_river.groupby('clean_address', sort=False)
              for f in flag_by_counting(list(rows['Date Opened']))]

def approach_3_self_merge():
    left = red_river[['clean_address', 'Request Number', 'Date Opened', 'window_start']]
    pairs = left.merge(red_river[['clean_address', 'Date Opened']].rename(columns={'Date Opened': 'other'}),
                       on='clean_address')
    pairs = pairs[(pairs['other'] <= pairs['Date Opened']) & (pairs['other'] >= pairs['window_start'])]
    return pairs.groupby('Request Number').size() >= 3

def approach_4_rolling():
    return (red_river.set_index('Date Opened').groupby('clean_address')['pickup_number']
                     .rolling('182D').count() >= 3)

def approach_5_shift():
    return (red_river.groupby('clean_address')['Date Opened'].shift(2)
            >= red_river['Date Opened'] - pd.DateOffset(months=6))

def seconds(fn):
    start = time.perf_counter(); fn(); return time.perf_counter() - start

sample = red_river.sample(500, random_state=1)       # approach 1 is too slow for every row
t1 = seconds(lambda: approach_1_loop_rows(sample)) * len(red_river) / len(sample)
pair_rows = int((red_river['clean_address'].value_counts() ** 2).sum())   # rows the merge creates
timing = pd.DataFrame({
    'seconds (all 12,946 rows)': [t1, seconds(approach_2_groupby_loop), seconds(approach_3_self_merge),
                                  seconds(approach_4_rolling), seconds(approach_5_shift)],
}, index=['1. iterrows loop (estimated from 500 rows)', '2. groupby + Python loop', '3. self-merge',
          '4. groupby().rolling("182D")', '5. groupby().shift(2)  (chosen)'])
timing['times slower than #5'] = timing.iloc[:, 0] / timing.iloc[-1, 0]
print(f'Self-merge builds {pair_rows:,} address pairs from {len(red_river):,} rows before filtering')
timing.style.format({'seconds (all 12,946 rows)': '{:.3f}', 'times slower than #5': '{:,.0f}x'})

Self-merge builds 35,052 address pairs from 12,946 rows before filtering


,"seconds (all 12,946 rows)",times slower than #5
1. iterrows loop (estimated from 500 rows),23.332,"4,068x"
2. groupby + Python loop,1.346,235x
3. self-merge,0.023,4x
"4. groupby().rolling(""182D"")",0.230,40x
5. groupby().shift(2) (chosen),0.006,1x


**What the timing shows.** Your seconds will differ by computer, but the ratios hold.

* **`shift(2)` is the fastest:** about 0.01 seconds for every row.
* **The `iterrows` loop that Claude listed first would take about 20+ seconds**, thousands of times slower, because every row re-scans the whole table. It's the solution the assignment warns about.
* **[REVIEW] Claude's ranking was partly wrong.** It ranked the self-merge as slower than `rolling`, but on this data the self-merge is about **8 times faster than `rolling`**. Its weakness is memory, which grows with the square of the misses per address. Here the worst address has only 21 misses, so the merge builds just 35,052 pairs. With a few addresses holding thousands of rows, the ranking would flip. The lesson: **time the options on the real data instead of trusting a ranking**, Claude's included.

In [31]:
# [ME] Add the challenge-rule fines to the saved results file
by_address = (red_river.assign(is_500=red_river['challenge_fine'] == 500)
                       .groupby('clean_address')
                       .agg(challenge_500_fines=('is_500', 'sum'),
                            challenge_damages=('challenge_fine', 'sum')))
fines_by_address = (fines_by_address.join(by_address)
                                    .sort_values(['challenge_damages', 'missed_pickups'], ascending=False))
fines_by_address.to_csv('results/red_river_fines_by_address.csv')
print('Challenge damages add up:', fines_by_address['challenge_damages'].sum() == challenge_total)
fines_by_address.head(10)

Challenge damages add up: True


,missed_pickups,fined_pickups,damages,first_miss,last_miss,challenge_500_fines,challenge_damages
clean_address,,,,,,,
12546 OLD HICKORY BLVD,21,20,4000,2018-08-17,2019-06-11,19,9700
5135 HICKORY HOLLOW PKWY,18,17,3400,2018-07-16,2019-06-18,16,8200
1584 BELL RD,19,18,3600,2017-11-27,2019-08-30,15,8100
3710 N NATCHEZ CT,16,15,3000,2017-11-29,2019-03-21,14,7200
607 ESTES RD,16,15,3000,2019-02-01,2019-10-23,14,7200
802 CRESCENT RD,16,15,3000,2019-04-03,2019-10-25,14,7200
1537 HARDING PL,15,14,2800,2018-03-08,2019-10-16,12,6400
206 ROLLING FORK CT,14,13,2600,2018-08-23,2019-10-10,12,6200
617 KINSEY BLVD,14,13,2600,2018-10-17,2019-08-20,12,6200


---
## Part 4 report (challenge rule)
| Result | Base rule (Part 3) | Challenge rule |
|---|---:|---:|
| Missed pickups (Red River) | 12,946 | 12,946 |
| $0 (first miss at an address) | 8,246 | 8,246 |
| $200 fines | 4,700 | 3,101 |
| **$500 fines** | none | **1,599** |
| **Total damages** | **$940,000** | **$1,419,700** |

**What it means:** the challenge rule adds **$479,700 (+51%)**. Each $500 replaces a $200, so 1,599 × $300 extra = $479,700. The $500 fines fall on **670 addresses**. Those are places where Red River missed the same house **three or more times within 6 months**, the chronic problem the contract targets. The worst, 12546 Old Hickory Blvd, goes from $4,000 to **$9,700**.

**Chosen method:** `groupby('clean_address')['Date Opened'].shift(2)`, compared with `date − 6 calendar months`. A pickup is 3rd-in-6-months exactly when the pickup two before it at the same address is still inside the window.

**Why I trust it:**

* **Three independent methods agree row for row:** `shift(2)`, `rolling('182D')`, and a plain brute-force loop.
* **A hand walk-through matches the code exactly:** 1005 Noelton Ave owes $2,100.
* **Five rules hold everywhere:** first misses are always $0; $500 only appears from the 3rd miss on; no address owes less than under the base rule; addresses with 2 or fewer misses are unchanged; and the extra money equals $300 × the number of $500 fines.
* **The window definition changes the total by at most $5,100.**

**Same caveats as Part 3:**

* If blank haulers are **not** filled in from the same address, the challenge total is **$1,312,600**, so the range is $1.31M to $1.42M.
* It is still a floor, because complaints and whole-street misses aren't fully counted.

The full per-address results, base and challenge, are in [`results/red_river_fines_by_address.csv`](results/red_river_fines_by_address.csv).